# Retrieval lab: how CourseMate finds the right notes

Run the cells from top to bottom with Shift+Enter.

In [1]:
# Setup: find the repo root, choose an embedder, and show which one we got.
import os
import sys
from pathlib import Path

# If Jupyter was started inside notebooks/, the repo root is the parent folder.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# Make "import coursemate" work even if the package is not installed.
sys.path.insert(0, str(ROOT))

from coursemate.embeddings import FakeEmbedder, OllamaEmbedder  # noqa: E402

# Read the provider from an environment variable; default to the real local model.
PROVIDER = os.environ.get("COURSEMATE_PROVIDER", "ollama")

if PROVIDER == "fake":
    embedder = FakeEmbedder()  # hash-based vectors, no model needed
else:
    embedder = OllamaEmbedder()  # needs Ollama running with nomic-embed-text

print("Repo root:  ", ROOT)
print("Provider:   ", PROVIDER)
print("Model:      ", embedder.model)
print("Dimensions: ", embedder.dim)

Repo root:   C:\Users\rawad\Downloads\coursemate
Provider:    ollama
Model:       nomic-embed-text
Dimensions:  768


## Part A. What an embedding looks like

An embedding turns a sentence into a fixed-length list of numbers, so that sentences with similar meaning end up with similar lists.

In [2]:
# Four sentences: s0 and s1 mean almost the same thing, s2 and s3 are unrelated.
sentences = [
    "Late projects lose 10% per day.",  # s0
    "How much is the penalty for handing in work after the deadline?",  # s1
    "Merge sort runs in O(n log n) time.",  # s2
    "Office hours are on Tuesday afternoons.",  # s3
]

# Turn every sentence into a vector (a list of floats).
vectors = embedder.embed_documents(sentences)

# Every vector has the same length, and that length is the model's dimension.
print("Length of one vector:", len(vectors[0]))

# Look at just the first 5 numbers of the first vector, rounded for readability.
print("First 5 numbers:", [round(x, 3) for x in vectors[0][:5]])

Length of one vector: 768
First 5 numbers: [0.048, 0.0, -0.159, 0.004, 0.018]


In [3]:
# Cosine similarity: close to 1 means "same direction" (similar meaning), near 0 means unrelated.
from coursemate.embeddings import cosine

labels = ["s0", "s1", "s2", "s3"]

# Header row with the column labels.
print("    " + "".join(f"{label:>8}" for label in labels))

# One row per sentence: compare it with every sentence (including itself).
for i, row_label in enumerate(labels):
    row = f"{row_label:>4}"
    for j in range(len(labels)):
        row += f"{cosine(vectors[i], vectors[j]):8.3f}"
    print(row)

          s0      s1      s2      s3
  s0   1.000   0.628   0.566   0.589
  s1   0.628   1.000   0.573   0.596
  s2   0.566   0.573   1.000   0.572
  s3   0.589   0.596   0.572   1.000
